# T3 gallery export · Six validation pairs for the showcase website (Kaggle T4)

Task **T3**, item 4 of the website's work list (`Progress_Tracker.md`, "Needs you now"). Built 17 Nov 2026. It trains nothing. It writes the gallery files the website reads, in the shape of `Essential_task_delegation/T3_Website/DATA_CONTRACT.md`.

**What it makes, for each of six validation pairs**
- the before-photo, the after-photo and the true damage mask;
- two damage maps, one per method:
  - **`before-after-unet`:** R11, our U-Net that reads the after-photo and the before-photo together (6 input channels, a ResNet-50 encoder), with 8-view test-time augmentation (TTA: the average of 8 flipped and rotated views). It runs here, on the GPU.
  - **`find-then-grade`:** SAM 3 outlines each building on the before-photo, and our grader labels each outline intact, damaged, destroyed or "not a building". The grader is a Siamese ResNet-18 that compares the before and after crops of one outline.
    - Nothing of it runs here: it reads E3's saved SAM 3 maps and R16c step 5's saved grades (grader seed 42, with its label biases).
    - Seed 42 is the one E7 fixed on 4 Nov for the building table.
- from each map:
  - the building table (H2's columns, plus each building's outline);
  - the survivability heatmap's three maps, hot spots and safe areas (T1 v3's code, unchanged);
  - the map's IoU per class against the truth (IoU, intersection over union: the overlap between the pixels a map gives a class and the pixels truly of it, from 0 to 1).

**Why:** the website shows four stand-in pairs today, with the weaker after-image-only U-Net (R9). These six are the real gallery. Each shows our better map, with find-then-grade beside it, on pairs no model trained on.

**Which pairs: the rule (fixed 17 Nov, before this notebook ran)**
1. **Candidates:** the 605 clean validation pairs, with at least 5,000 px of truly destroyed buildings, so every pair shows the hardest class.
   - The 605 are the 645 validation pairs minus the 40 that share images with training (E5).
2. **One event per disaster type, for range:** hurricane, wildfire, tornado, tsunami, flood and earthquake. Within each type, the event with the most candidates; a tie is broken by destroyed area.
3. **Within each event, the typical pair:** the median of R11's destroyed-building IoU among its candidates, not the best and not the worst.
   - For an even count, the lower middle.
   - Ties go to the larger destroyed area.
4. **The Home page** opens on the pair with the largest destroyed area outside Turkey.

R11's per-image scores for the rule come from E5 step 2 (`E5-results/step2/per_image_counts.csv`, R11 with TTA). The result:

| Pair (after-photo) | Type | Candidates | R11 destroyed IoU | Find-then-grade destroyed IoU | True destroyed px |
|---|---|---|---|---|---|
| hurricane_harvey_00000472_post_disaster.png | hurricane | 5 | 0.000 | 0.000 | 14,369 |
| santa_rosa_wildfire_00000058_post_disaster.png (Home) | wildfire | 7 | 0.752 | 0.762 | 70,825 |
| moore_tornado_00000120_post_disaster.png | tornado | 5 | 0.659 | 0.531 | 25,405 |
| palu_tsunami_00000122_post_disaster.png | tsunami | 2 | 0.277 | 0.314 | 12,197 |
| midwest_flooding_00000287_post_disaster.png | flood | 1 | 0.000 | 0.000 | 53,449 |
| turkey_earthquake0_post_380.png | earthquake | 13 | 0.000 | 0.460 | 31,140 |

- **R11 finds none of the destroyed buildings in three of the six.** Every Harvey and Turkey candidate scores 0 with R11, so a typical pick can't avoid it. That's the honest spread.
- **Find-then-grade** finds 46% of the Turkey pair's destroyed area, but misses Harvey's and the flood's too.
- Find-then-grade's column was computed on the PC from the same saved files, before this notebook ran; Cell 5 must reproduce it.

**Checks (the notebook stops before writing anything if one fails)**
1. The truth rebuilt here from the dataset's three masks equals E1's saved truth: the same pixel count per class.
2. The truth under each SAM 3 outline equals what R16c step 5 saved for that outline.
3. R11 reproduces its saved results:
   - its IoU per class within 0.002 of E5 step 2's;
   - its pixels under the outlines within 2% of what R16c step 5 saved (E7's limit).
4. Find-then-grade's IoU per class equals the table above, to 4 decimals.
5. Each pair's files follow the website's rules that can be checked here. On the PC, `node scripts/validate-data.ts` checks them in full.

**How to run on Kaggle** (account 1: R11's checkpoint is private, and E3's and R16c step 5's outputs are there)
1. **Settings:** GPU T4 x2 (one is used), internet ON, secret `HF_TOKEN` (AbrarAlam, read access is enough).
2. **Inputs:**
   - `abrarmohammedtanzim/disasterm3-mirror`;
   - Your Work → `e3-sam3-building-finding`;
   - Your Work → `r16c-step5-grader-on-sam3-outlines`.
3. **Smoke test:** Run All once with `SMOKE_TEST = True` in Cell 2 (the first pair only, a few minutes). If Cell 1 says to restart the kernel, restart and Run All again.
4. **The run:** set `SMOKE_TEST = False` and use Save & Run All (Commit). About 10 minutes, estimated.

**Outputs** in `/kaggle/working/gallery/`, and the same as `/kaggle/working/t3_gallery.zip`:
- one folder per pair, as in the contract;
- `README.md` and `config.json`: the rule, the checks and the versions.

**On a PC**, this notebook does a dry run instead:
- the truth comes from the local masks, and find-then-grade from the local files;
- E1's saved R9 maps stand in for R11, since there's no GPU;
- everything is marked as a fixture, so the website's validator checks it with `--fixtures`.

A dry run's files must never go into `public/data/`.


In [1]:
# ── Cell 1: Environment (Run-All safe): R11 needs segmentation-models-pytorch 0.3.4 ──
# R11's checkpoint has 0.3.4's key layout, as in E1, E5 and E7. Nothing else is installed: SAM 3 and the grader
# don't run here (their saved outlines and grades are read), so this notebook needs no SAM 3 access.
import importlib.metadata as _md
import os
import subprocess
import sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()
PINS = {"segmentation-models-pytorch": "0.3.4"}


def _mismatched():
    out = []
    for pkg, want in PINS.items():
        try:
            have = _md.version(pkg)
        except _md.PackageNotFoundError:
            have = "not installed"
        if have != want:
            out.append(f"{pkg}: {have} → {want}")
    return out


if not ON_KAGGLE:
    print("✓ On a PC: a dry run, with no model and no installs (see the first cell)")
elif _mismatched():
    print("⏳ Installing pinned versions:")
    for _m in _mismatched():
        print(f"   {_m}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *(f"{k}=={v}" for k, v in PINS.items())], check=True)
    if _mismatched():
        raise RuntimeError(f"Pins still wrong after install: {_mismatched()}")
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
        print("✓ Dependencies installed (background run): continuing without a restart.")
    else:
        raise SystemExit("✓ Dependencies installed. RESTART THE KERNEL NOW "
                         "(Run → Restart & clear cell outputs), then Run All again.")
else:
    print("✓ All pinned versions already installed: proceeding.")

⏳ Installing pinned versions:
   segmentation-models-pytorch: not installed → 0.3.4
✓ Dependencies installed (background run): continuing without a restart.


In [2]:
# ── Cell 2: Configuration and inputs ──────────────────────────────────────
import datetime
import json
import math
import re
import shutil
import zipfile
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage

SMOKE_TEST = False                 # True = the first pair only
DRY_RUN = not ON_KAGGLE            # on a PC: no model; E1's R9 maps stand in for R11, and everything is marked a fixture
NOTEBOOK = "Essential_task_delegation/Champion_Task/t3-gallery-export.ipynb"
UNET, HF_USER = "R11", "AbrarAlam"    # checkpoint: AbrarAlam/disasterm3-d4-R11/final_model.pth (private)
TABLE_SEED = 42                    # the grader seed E7 fixed on 4 Nov for the building table
LABELS = ["Intact", "Damaged", "Destroyed", "None"]                  # the grader's labels
BIAS_FIXED = [0.0, -1.75, -4.0, -1.25]                               # seed 42's biases (R16c step 5, calibration.json)
NUM_CLASSES, MIN_AREA = 4, 20      # 0 background, 1 intact, 2 damaged, 3 destroyed; H2's smallest building
OUTLINE_TOLERANCE_PX = 0.75        # as the website's buildingsFromMask (src/core/buildings.ts)
WEBP_QUALITY = 82                  # as the website's fixtures
LIMITS = {"r11_iou": 0.002, "r11_px": 0.02, "ftg_iou": 0.0001, "truth_px_share": 0.0}   # the checks in the first cell
CLASS_KEYS = ["background", "intact", "damaged", "destroyed"]

# The six pairs from the rule in the first cell. "expect" is what this notebook must reproduce: the truth's pixels per
# class (E1's saved truth), R11's IoU per class (E5 step 2, with TTA) and find-then-grade's (computed on the PC, 17 Nov).
PAIRS = [
    {"image": "hurricane_harvey_00000472_post_disaster.png", "type": "hurricane", "candidates": 5,
     "expect": {"truth_px": [924432, 0, 109775, 14369], "r11_iou": [0.9562, None, 0.6616, 0.0],
                "ftg_iou": [0.9546, 0.0, 0.5871, 0.0]}},
    {"image": "santa_rosa_wildfire_00000058_post_disaster.png", "type": "wildfire", "candidates": 7, "hero": True,
     "expect": {"truth_px": [744246, 233505, 0, 70825], "r11_iou": [0.9006, 0.795, 0.0, 0.7517],
                "ftg_iou": [0.9234, 0.8246, 0.0, 0.7619]}},
    {"image": "moore_tornado_00000120_post_disaster.png", "type": "tornado", "candidates": 5,
     "expect": {"truth_px": [1014876, 0, 8295, 25405], "r11_iou": [0.9928, 0.0, 0.4001, 0.6592],
                "ftg_iou": [0.9889, 0.0, 0.366, 0.5311]}},
    {"image": "palu_tsunami_00000122_post_disaster.png", "type": "tsunami", "candidates": 2,
     "expect": {"truth_px": [781314, 253942, 1123, 12197], "r11_iou": [0.8927, 0.6914, 0.0, 0.2769],
                "ftg_iou": [0.9077, 0.6992, 0.0888, 0.3142]}},
    {"image": "midwest_flooding_00000287_post_disaster.png", "type": "flood", "candidates": 1,
     "expect": {"truth_px": [995127, 0, 0, 53449], "r11_iou": [0.9716, None, 0.0, 0.0],
                "ftg_iou": [0.9496, None, 0.0, 0.0]}},
    {"image": "turkey_earthquake0_post_380.png", "type": "earthquake", "candidates": 13,
     "expect": {"truth_px": [696943, 320493, 0, 31140], "r11_iou": [0.8511, 0.6495, None, 0.0],
                "ftg_iou": [0.8536, 0.699, None, 0.4596]}},
]
if SMOKE_TEST:
    PAIRS = PAIRS[:1]

if ON_KAGGLE:
    INPUT_ROOT = Path("/kaggle/input")
    DATA_ROOT = INPUT_ROOT / "datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct"
    MANIFEST_PATH = DATA_ROOT / "train_release.json"
    OUT_ROOT = Path("/kaggle/working")
else:                              # run from Essential_task_delegation/Champion_Task, as T1 v3 does on a PC
    REPO = Path.cwd().resolve().parents[1]
    INPUT_ROOT = Path.cwd()
    DATA_ROOT = REPO / "dataset" / "DisasterM3 Dataset" / "DisasterM3_Instruct"
    MANIFEST_PATH = REPO / "train_release.json"
    import tempfile                # a dry run writes outside the repository unless told otherwise
    OUT_ROOT = Path(os.environ.get("GALLERY_DRY_RUN_OUT", Path(tempfile.gettempdir()) / "t3-gallery-dry-run")).resolve()
    XBD_DIRS = [REPO / "data" / "xView2 Challenge Dataset - train and test" / s / "images" for s in ("train", "test")]
    R9_STAND_IN = Path("E1-results/E1/preds/R9_tta")
OUT_DIR = OUT_ROOT / "gallery"

# One walk over the inputs: the files and folders this notebook looks for
FOUND, FOUND_DIRS = defaultdict(list), defaultdict(list)
for root, dirs, files in os.walk(INPUT_ROOT if ON_KAGGLE else Path("R16c-results/step5")):
    for name in files:
        if name in ("instances_val.csv", "calibration.json"):
            FOUND[name].append(Path(root) / name)
if ON_KAGGLE:
    for root, dirs, files in os.walk(INPUT_ROOT):
        for name in dirs:
            if name == "pre_grid2":
                FOUND_DIRS[name].append(Path(root) / name)
else:
    FOUND_DIRS["pre_grid2"] = [Path("E3-results/E3/masks/pre_grid2")]

STEP5_WHY = "attach Your Work → r16c-step5-grader-on-sam3-outlines"
INST_PATH = next((p for p in sorted(FOUND["instances_val.csv"])
                  if {f"logp_{l.lower()}_s{TABLE_SEED}" for l in LABELS} <= set(pd.read_csv(p, nrows=0).columns)), None)
if INST_PATH is None:
    raise FileNotFoundError(f"No instances_val.csv with the seed-{TABLE_SEED} grader's four labels: {STEP5_WHY}")
CALIB_PATH = next((p for p in sorted(FOUND["calibration.json"])
                   if str(TABLE_SEED) in json.loads(p.read_text(encoding="utf-8"))
                   and [json.loads(p.read_text(encoding="utf-8"))[str(TABLE_SEED)]["bias"].get(l) for l in LABELS] == BIAS_FIXED),
                  None)
if CALIB_PATH is None:
    raise FileNotFoundError(f"No calibration.json with seed {TABLE_SEED}'s biases {BIAS_FIXED}: {STEP5_WHY}")
if not FOUND_DIRS["pre_grid2"] or not FOUND_DIRS["pre_grid2"][0].is_dir():
    raise FileNotFoundError("No pre_grid2 folder (SAM 3's saved maps): attach Your Work → e3-sam3-building-finding")
E3_DIR = sorted(FOUND_DIRS["pre_grid2"])[0]
assert MANIFEST_PATH.exists(), f"No manifest at {MANIFEST_PATH}: attach abrarmohammedtanzim/disasterm3-mirror"

shutil.rmtree(OUT_DIR, ignore_errors=True)
OUT_DIR.mkdir(parents=True)
TODAY = datetime.date.today().isoformat()
print(f"✓ T3 gallery export{'  [SMOKE TEST]' if SMOKE_TEST else ''}{'  [DRY RUN on a PC]' if DRY_RUN else ''}: {len(PAIRS)} pair(s)")
print(f"  SAM 3 maps: {E3_DIR}")
print(f"  Grades:     {INST_PATH} (seed {TABLE_SEED}); biases from {CALIB_PATH}")
print(f"  Output:     {OUT_DIR}")

✓ T3 gallery export: 6 pair(s)
  SAM 3 maps: /kaggle/input/notebooks/abrarmohammedtanzim/e3-sam3-building-finding/E3/masks/pre_grid2
  Grades:     /kaggle/input/notebooks/abrarmohammedtanzim/r16c-step5-grader-on-sam3-outlines/step5/instances_val.csv (seed 42); biases from /kaggle/input/notebooks/abrarmohammedtanzim/r16c-step5-grader-on-sam3-outlines/step5/calibration.json
  Output:     /kaggle/working/gallery


In [3]:
# ── Cell 3: Helpers (pure functions), with self-checks ──────────────────────
XBD_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster")
OTHER_NAME = re.compile(r"^(.+?)_post")


def name_of(path):
    return Path(str(path).replace("\\", "/")).name


def event_and_source(name):
    """E1's rule: 'hurricane_florence_00000261_post_disaster.png' -> ('hurricane_florence', 'xBD'),
    'turkey_earthquake5_post_12.png' -> ('turkey_earthquake', 'other')."""
    stem = Path(name).stem
    m = XBD_NAME.match(stem)
    if m:
        return m.group(1), "xBD"
    m = OTHER_NAME.match(stem)
    return re.sub(r"\d+$", "", m.group(1) if m else stem), "other"


def resolve_path(rel_path):
    """The D4 notebooks' path rule for the mirror's images and masks."""
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    for c in (DATA_ROOT / rel_path, DATA_ROOT / "train_images" / rel_path, DATA_ROOT / "train_images" / "train_images" / filename,
              DATA_ROOT / "masks" / rel_path, DATA_ROOT / "masks" / "masks" / filename):
        if c.exists():
            return c
    return None


def read_rgb(path):
    image = cv2.imread(str(path))
    return None if image is None else cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


FOLDER_TO_CLASS = {"train_building_intact_mask": 1, "train_building_damaged_mask": 2, "train_building_destroyed_mask": 3}


def truth_from_masks(mask_paths, shape):
    """The D4 notebooks' rule: intact, then damaged, then destroyed, each written over the last; resized to the photo
    with nearest-neighbour if the sizes differ, as their dataset class does."""
    truth = None
    for folder, cls in FOLDER_TO_CLASS.items():
        if folder not in mask_paths:
            continue
        m = cv2.imread(str(mask_paths[folder]), cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if truth is None:
            truth = np.zeros_like(m, dtype=np.uint8)
        truth[m > 0] = cls
    if truth is not None and shape is not None and truth.shape != shape:
        truth = cv2.resize(truth, (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST)
    return truth


def round1(v):
    """JavaScript's Math.round(v * 10) / 10, as the website rounds (half up, not Python's half to even)."""
    return math.floor(v * 10 + 0.5) / 10


def trace_rings(inside, x0, y0, w, h):
    """Outline rings of one region along pixel edges, outer ring first: a line-for-line port of the website's
    traceRings (src/core/buildings.ts), so the gallery's outlines match its in-browser ones."""
    w1 = w + 2

    def key(x, y):
        return (y - y0) * w1 + (x - x0)

    nxt = {}

    def add(ax, ay, bx, by):
        nxt.setdefault(key(ax, ay), []).append((bx, by))

    for y in range(y0, y0 + h):
        for x in range(x0, x0 + w):
            if not inside(x, y):
                continue
            if not inside(x, y - 1):
                add(x, y, x + 1, y)
            if not inside(x + 1, y):
                add(x + 1, y, x + 1, y + 1)
            if not inside(x, y + 1):
                add(x + 1, y + 1, x, y + 1)
            if not inside(x - 1, y):
                add(x, y + 1, x, y)
    rings = []
    for start_key, lst in nxt.items():
        while lst:
            sx, sy = start_key % w1 + x0, start_key // w1 + y0
            ring = [[sx, sy]]
            cx, cy = lst.pop()
            dx, dy = cx - sx, cy - sy
            while (cx, cy) != (sx, sy):
                out = nxt[key(cx, cy)]
                pick = 0
                if len(out) > 1:          # pixels touching only at a corner: turn left, as for 8-connected regions
                    pick = next((i for i, (nx, ny) in enumerate(out) if nx - cx == dy and ny - cy == -dx), 0)
                nx, ny = out.pop(pick)
                ndx, ndy = nx - cx, ny - cy
                if (ndx, ndy) != (dx, dy):
                    ring.append([cx, cy])  # corners only
                dx, dy, cx, cy = ndx, ndy, nx, ny
            rings.append(ring)

    def area(r):
        return abs(sum(x * r[(i + 1) % len(r)][1] - r[(i + 1) % len(r)][0] * y for i, (x, y) in enumerate(r)))

    return sorted(rings, key=lambda r: -area(r))


def simplify_ring(ring, tol):
    """Douglas-Peucker on a ring: a port of the website's simplifyRing."""
    if len(ring) <= 4 or tol <= 0:
        return ring
    keep = [0] * len(ring)
    x0, y0 = ring[0]
    far = 0
    for i, p in enumerate(ring):
        if math.hypot(p[0] - x0, p[1] - y0) > math.hypot(ring[far][0] - x0, ring[far][1] - y0):
            far = i
    keep[0] = keep[far] = 1
    stack = [(0, far), (far, len(ring))]
    while stack:
        a, b = stack.pop()
        ax, ay = ring[a]
        bx, by = ring[b % len(ring)]
        ln = math.hypot(bx - ax, by - ay) or 1
        worst, worst_d = -1, tol
        for i in range(a + 1, b):
            d = abs((bx - ax) * (ay - ring[i][1]) - (ax - ring[i][0]) * (by - ay)) / ln
            if d > worst_d:
                worst_d, worst = d, i
        if worst >= 0:
            keep[worst] = 1
            stack.extend([(a, worst), (worst, b)])
    out = [p for p, k in zip(ring, keep) if k]
    return out if len(out) >= 3 else ring


CLASS_NAME = {1: "intact", 2: "damaged", 3: "destroyed"}


def buildings_from_mask(mask, image, tol=OUTLINE_TOLERANCE_PX):
    """H2's table (8-connected regions of one class, at least MIN_AREA px, centre-to-centre distance to the nearest other
    destroyed building), in the contract's columns, with each region's outline. Rows in the website's order: by class,
    then by each region's first pixel in reading order."""
    rows = []
    for cls in (1, 2, 3):
        n, labels, stats, cents = cv2.connectedComponentsWithStats((mask == cls).astype(np.uint8), connectivity=8)
        if n <= 1:
            continue
        flat = labels.ravel()
        first = np.full(n, flat.size, np.int64)
        np.minimum.at(first, flat, np.arange(flat.size))
        for k in sorted(range(1, n), key=lambda k: first[k]):
            area = int(stats[k, cv2.CC_STAT_AREA])
            if area < MIN_AREA:
                continue
            bx, by = int(stats[k, cv2.CC_STAT_LEFT]), int(stats[k, cv2.CC_STAT_TOP])
            bw, bh = int(stats[k, cv2.CC_STAT_WIDTH]), int(stats[k, cv2.CC_STAT_HEIGHT])
            region = np.zeros((bh + 2, bw + 2), bool)
            region[1:-1, 1:-1] = labels[by:by + bh, bx:bx + bw] == k

            def inside(x, y, r=region, bx=bx, by=by):
                return bool(r[y - by + 1, x - bx + 1])

            rings = [simplify_ring(r, tol) for r in trace_rings(inside, bx, by, bw, bh)]
            rows.append({"image": image, "building_id": len(rows) + 1, "class_id": cls, "class_name": CLASS_NAME[cls],
                         "area_px": area, "centroid_x": round1(float(cents[k][0])), "centroid_y": round1(float(cents[k][1])),
                         "bbox_x": bx, "bbox_y": by, "bbox_w": bw, "bbox_h": bh,
                         "dist_to_nearest_destroyed_px": None, "outline": rings})
    destroyed = [b for b in rows if b["class_id"] == 3]
    for b in rows:
        best = min((math.hypot(d["centroid_x"] - b["centroid_x"], d["centroid_y"] - b["centroid_y"])
                    for d in destroyed if d is not b), default=None)
        b["dist_to_nearest_destroyed_px"] = None if best is None else round1(best)
    return rows


def iou_per_class(pred, truth):
    """IoU per class (background, intact, damaged, destroyed), 4 decimals; None when a class is in neither mask."""
    cm = np.bincount(truth.astype(np.int64).ravel() * NUM_CLASSES + pred.astype(np.int64).ravel(),
                     minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)
    out = []
    for c in range(NUM_CLASSES):
        union = int(cm[c, :].sum() + cm[:, c].sum() - cm[c, c])
        out.append(None if union == 0 else round(float(cm[c, c] / union), 4))
    return out


def grades_of(rows, biases):
    """Each outline's class from the grader's saved log-probabilities and its biases: intact, damaged, destroyed -> 1-3,
    "None" (not a building) -> 0 (background)."""
    logp = rows[[f"logp_{l.lower()}_s{TABLE_SEED}" for l in LABELS]].to_numpy(np.float64)
    label = np.argmax(logp + np.asarray(biases), 1)
    return np.where(label < 3, label + 1, 0).astype(np.uint8)


def find_then_grade_map(sam_ids, rows, biases):
    """SAM 3's outlines filled with each outline's grade, 0 elsewhere (the contract's find-then-grade mask)."""
    lut = np.zeros(int(sam_ids.max()) + 1, np.uint8)
    ids = rows["id"].to_numpy()
    keep = ids <= sam_ids.max()
    lut[ids[keep]] = grades_of(rows, biases)[keep]
    return lut[sam_ids]


def pixels_under_outlines(sam_ids, mask):
    """Pixels per class (0-3) under each outline id: rows indexed by id."""
    size = int(sam_ids.max()) + 1
    return np.bincount(sam_ids.ravel().astype(np.int64) * NUM_CLASSES + mask.ravel(),
                       minlength=size * NUM_CLASSES).reshape(size, NUM_CLASSES)


# ── T1 v3's heatmap code, unchanged (t1-survivability-heatmap-v3.ipynb, Cells 1-3) ──
SIGMA_PX = 25                  # danger-map blur (kernel density bandwidth), as in v1
DAMAGED_WEIGHT = 0.5           # destroyed pixels weigh 1.0
SAFE_SCALE_PX = 200            # this far from any damage counts as fully safe
HOT_THRESHOLD = 0.05           # search priority above this forms a hot zone
MIN_SPOT_PX = 500              # v2: zones smaller than this are specks, not hot spots
POINT_EDGE_MARGIN_PX = 32      # v3: a hot spot's point stays this far from the image edge when its zone allows
SAFE_MIN_GAP_PX = 150          # the 3 safest areas are at least this far apart
SAFE_EDGE_MARGIN_PX = 100      # v2: safe areas stay this far from the image edge
POINT_RADIUS_PX = 20           # v2: a marked point is "on" a building if one is this close (a few metres)
EIGHT = np.ones((3, 3), bool)


def survivability_maps(mask):
    """Returns danger, safety and priority (float32, 0-1), and the distance in px to the nearest damage."""
    weight = (mask == 3).astype(np.float32) + DAMAGED_WEIGHT * (mask == 2)
    danger = ndimage.gaussian_filter(weight, SIGMA_PX, mode="mirror", truncate=4.0)   # = v1's cv2.GaussianBlur
    damage = mask >= 2
    dist = (ndimage.distance_transform_edt(~damage).astype(np.float32) if damage.any()
            else np.full(mask.shape, np.inf, np.float32))
    safety = np.minimum(dist / SAFE_SCALE_PX, 1).astype(np.float32)
    return danger, safety, danger * (1 - safety), dist


def compass(x, y, w, h):
    """Where a point sits in the image (top = north): 'north-east', 'west', 'centre', ..."""
    row = ("north", "", "south")[min(int(3 * y / h), 2)]
    col = ("west", "", "east")[min(int(3 * x / w), 2)]
    return "-".join(p for p in (row, col) if p) or "centre"


def hot_spots(priority, k=3):
    """The k strongest zones (connected, priority >= HOT_THRESHOLD, at least MIN_SPOT_PX pixels), ranked by
    total priority. Each is marked at its strongest point at least POINT_EDGE_MARGIN_PX from the image edge
    (anywhere in the zone if none of it is that far in), the one nearest the zone's middle if several tie.
    Also returns the label image ('label' is each zone's id in it)."""
    labels, n = ndimage.label(priority >= HOT_THRESHOLD, structure=EIGHT)
    if n == 0:
        return [], labels
    area = np.bincount(labels.ravel(), minlength=n + 1)
    mass = np.bincount(labels.ravel(), weights=priority.ravel(), minlength=n + 1)
    mass[0] = -1                                    # label 0 is everything below the threshold
    mass[area < MIN_SPOT_PX] = -1                   # specks
    h, w = priority.shape
    m = POINT_EDGE_MARGIN_PX
    spots = []
    for i in np.argsort(-mass)[:k]:
        if mass[i] < 0:
            break
        ys, xs = np.nonzero(labels == i)
        peak = float(priority[ys, xs].max())
        inner = (ys >= m) & (ys < h - m) & (xs >= m) & (xs < w - m)
        if inner.any():                             # v3: keep the mark off the image edge
            ys, xs = ys[inner], xs[inner]
        values = priority[ys, xs]
        top = values >= values.max() - 1e-6
        j = np.argmin((ys[top] - ys.mean()) ** 2 + (xs[top] - xs.mean()) ** 2)
        x, y = int(xs[top][j]), int(ys[top][j])
        spots.append({"rank": len(spots) + 1, "label": int(i), "point_xy": [x, y], "area_px": int(area[i]),
                      "peak": round(peak, 3), "edge_px": int(min(x, y, w - 1 - x, h - 1 - y)),
                      "where": compass(x, y, w, h)})
    return spots, labels


def safe_areas(dist, mask, k=3):
    """The k open (background) points farthest from any damage, at least SAFE_MIN_GAP_PX apart and at least
    SAFE_EDGE_MARGIN_PX from the image edge, because damage just outside the picture is invisible."""
    if np.isinf(dist).all():
        return []                                   # no damage predicted: the whole image is safe
    h, w = mask.shape
    m = SAFE_EDGE_MARGIN_PX
    inner = (slice(m, h - m), slice(m, w - m))
    score = np.full(mask.shape, -1.0)
    score[inner] = np.where(mask[inner] == 0, dist[inner], -1.0)
    yy, xx = np.ogrid[:h, :w]
    areas = []
    for rank in range(1, k + 1):
        y, x = np.unravel_index(np.argmax(score), score.shape)
        if score[y, x] < 0:
            break
        areas.append({"rank": rank, "xy": [int(x), int(y)], "distance_px": int(score[y, x]),
                      "edge_px": int(min(x, y, w - 1 - x, h - 1 - y)), "where": compass(x, y, w, h)})
        score[(yy - y) ** 2 + (xx - x) ** 2 < SAFE_MIN_GAP_PX ** 2] = -1
    return areas


def save_map(m, path):
    """256 x 256 copy (area average), value x 255."""
    small = np.asarray(Image.fromarray(np.clip(m, 0, 1).astype(np.float32)).resize((256, 256), Image.BOX))
    Image.fromarray(np.round(small * 255).astype(np.uint8)).save(path)


def near(gt, xy, classes):
    """Is one of these ground-truth classes within POINT_RADIUS_PX of the point?"""
    x, y, r = xy[0], xy[1], POINT_RADIUS_PX
    return bool(np.isin(gt[max(0, y - r):y + r + 1, max(0, x - r):x + r + 1], classes).any())


def heatmap_outputs(pred, gt, out_dir):
    """T1 v3's Cell 3 for one mask: the three maps (saved), hot spots and safe areas with the ground-truth checks."""
    danger, safety, priority, dist = survivability_maps(pred)
    spots, labels = hot_spots(priority)
    for s in spots:
        zone = labels == s.pop("label")
        s["on_gt_destroyed"] = bool((zone & (gt == 3)).any())          # v1's check: the zone touches it
        s["on_gt_damage"] = bool((zone & (gt >= 2)).any())
        s["point_on_gt_destroyed"] = near(gt, s["point_xy"], [3])      # v2: the marked point itself
        s["point_on_gt_damage"] = near(gt, s["point_xy"], [2, 3])
    safe = safe_areas(dist, pred)
    for a in safe:
        a["on_gt_building"] = bool(gt[a["xy"][1], a["xy"][0]] > 0)
    for kind, m in (("kde", danger), ("safe", safety), ("priority", priority)):
        save_map(m, out_dir / f"heat_{kind}.png")
    return {"hot_spots": spots, "safe_areas": safe}


def write_json(path, obj, compact=False):
    path.write_text(json.dumps(obj, separators=(",", ":")) if compact else json.dumps(obj, indent=1), encoding="utf-8")


def write_ids_png(path, ids):
    """One channel, 8-bit class ids."""
    assert ids.dtype == np.uint8 and ids.ndim == 2 and int(ids.max(initial=0)) <= 3
    if not cv2.imwrite(str(path), ids):
        raise OSError(f"couldn't write {path}")


# Self-checks
_m = np.zeros((200, 200), np.uint8)
_m[10:20, 10:20] = 1                    # intact, 100 px
_m[20, 20] = 1                          # touches that block on a diagonal: still one building
_m[100:102, 100:102] = 1                # intact, 4 px: under MIN_AREA
_m[50:60, 50:60] = 3                    # destroyed
_m[50:60, 150:160] = 3                  # destroyed
_m[150:153, 150:153] = 3                # destroyed, 9 px: under MIN_AREA
_b = buildings_from_mask(_m, "x.png", tol=0)
assert [(b["class_id"], b["area_px"]) for b in _b] == [(1, 101), (3, 100), (3, 100)]
assert [b["dist_to_nearest_destroyed_px"] for b in _b] == [56.4, 100.0, 100.0]
assert _b[0]["outline"][0][:3] == [[10, 10], [20, 10], [20, 20]] and _b[1]["outline"] == [[[50, 50], [60, 50], [60, 60], [50, 60]]]
_r = np.zeros((20, 20), np.uint8)
_r[2:12, 2:12] = 2
_r[5:8, 5:8] = 0                        # a hole: two rings, the outer one first
_rings = buildings_from_mask(_r, "x.png", tol=0)[0]["outline"]
assert len(_rings) == 2 and _rings[0] == [[2, 2], [12, 2], [12, 12], [2, 12]]
assert {(5, 5), (5, 8), (8, 5), (8, 8)} <= set(map(tuple, _rings[1]))     # the hole (its ring may start mid-edge, as on the site)
assert round1(0.25) == 0.3 and round1(0.35) == 0.4 and round1(2.0) == 2.0      # half up, as the website
assert iou_per_class(_m, _m) == [1.0, 1.0, None, 1.0]
_rows = pd.DataFrame({"id": [1, 2, 3], **{f"logp_{l.lower()}_s{TABLE_SEED}": v for l, v in
                      zip(LABELS, ([-0.1, -9, -9], [-9, -0.1, -9], [-9, -9, -0.1], [-3, -3, -3]))}})
assert grades_of(_rows, [0, 0, 0, 0]).tolist() == [1, 2, 3]
assert find_then_grade_map(np.array([[0, 1], [2, 3]]), _rows, [0, 0, 0, 0]).tolist() == [[0, 1], [2, 3]]
assert grades_of(_rows.assign(**{f"logp_none_s{TABLE_SEED}": [0, 0, 0]}), [0, 0, 0, 0]).tolist() == [0, 0, 0]
_hm = np.zeros((600, 600), np.uint8)    # T1 v3's own checks: one destroyed block -> one hot spot on it, safe areas far away
_hm[280:320, 280:320] = 3
_d, _s, _p, _dist = survivability_maps(_hm)
_spots, _ = hot_spots(_p)
assert len(_spots) == 1 and _spots[0]["where"] == "centre" and _hm[_spots[0]["point_xy"][1], _spots[0]["point_xy"][0]] == 3
assert len(safe_areas(_dist, _hm)) == 3
assert safe_areas(np.full((5, 5), np.inf, np.float32), np.zeros((5, 5), np.uint8)) == []
print("✓ Helpers ready (self-checks passed)")

✓ Helpers ready (self-checks passed)


In [4]:
# ── Cell 4: The pairs' photos, truth, SAM 3 outlines and grades; checks 1 and 2 ──
with open(MANIFEST_PATH, encoding="utf-8") as f:
    manifest = json.load(f)
WANTED = {p["image"] for p in PAIRS}
ENTRIES = defaultdict(dict)
for e in manifest:
    if (e.get("task") == "Referring Expression Segmentation" and e.get("cls_description") == "Building Damage Assessment"
            and e.get("image_type") == "Optical" and name_of(e.get("post_image_path", "")) in WANTED):
        entry = ENTRIES[name_of(e["post_image_path"])]
        folder = Path(e.get("ground_truth", "").replace("\\", "/")).parent.name   # e.g. train_building_intact_mask
        mask_path = resolve_path(e.get("ground_truth", ""))
        if mask_path:
            entry.setdefault("masks", {})[folder] = mask_path
        entry["post"], entry["pre"] = e["post_image_path"], e.get("pre_image_path")
del manifest
assert WANTED <= set(ENTRIES), f"Not in the manifest: {sorted(WANTED - set(ENTRIES))}"

INST = pd.read_csv(INST_PATH).query("found_by == 'sam3'")
BIASES = [json.loads(CALIB_PATH.read_text(encoding="utf-8"))[str(TABLE_SEED)]["bias"][l] for l in LABELS]
TRUTH_COLS = ["gt_background_px", "gt_intact_px", "gt_damaged_px", "gt_destroyed_px"]


def xbd_photo(name, when):
    """PC dry run only: xBD's original file for a DisasterM3 xBD pair, if this PC has it."""
    m = XBD_NAME.match(Path(name).stem)
    if not m:
        return None
    stem = f"{m.group(1).replace('_', '-')}_{m.group(2)}_{when}_disaster.png"
    return next((d / stem for d in XBD_DIRS if (d / stem).exists()), None)


DATA, CHECKS = {}, []


def check(image, what, result, passed):
    CHECKS.append({"image": image, "check": what, "result": result, "pass": bool(passed)})


for p in PAIRS:
    name, entry = p["image"], ENTRIES[p["image"]]
    if ON_KAGGLE:
        post_path, pre_path = resolve_path(entry["post"]), resolve_path(entry["pre"])
        assert post_path and pre_path, f"{name}: a photo isn't in the mirror ({entry['post']}, {entry['pre']})"
    else:
        post_path, pre_path = xbd_photo(name, "post"), xbd_photo(name, "pre")
    post = read_rgb(post_path) if post_path else None
    pre = read_rgb(pre_path) if pre_path else None
    if ON_KAGGLE:
        assert post is not None and pre is not None, f"{name}: a photo can't be read"
    pre_resized = pre is not None and post is not None and pre.shape != post.shape
    if pre_resized:                       # as E7 and the D4 dataset class: the before-photo at the after-photo's size
        pre = cv2.resize(pre, (post.shape[1], post.shape[0]), interpolation=cv2.INTER_LINEAR)
    truth = truth_from_masks(entry.get("masks", {}), post.shape[:2] if post is not None else None)
    assert truth is not None, f"{name}: no mask found for it in the mirror"
    shape = truth.shape

    # Check 1: the truth equals E1's saved truth, pixel count per class
    px = np.bincount(truth.ravel(), minlength=NUM_CLASSES).tolist()
    check(name, "1. truth = E1's saved truth (pixels per class)", f"{px} against {p['expect']['truth_px']}",
          px == p["expect"]["truth_px"])

    sam = cv2.imread(str(E3_DIR / f"{Path(name).stem}.png"), cv2.IMREAD_UNCHANGED)
    assert sam is not None, f"{name}: no SAM 3 map in {E3_DIR}"
    if sam.shape != shape:
        sam = cv2.resize(sam, (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST)
    sam = sam.astype(np.int64)
    rows = INST[INST["image"] == name]
    assert len(rows), f"{name}: no saved grades in {INST_PATH}"
    ungraded = sorted(set(np.unique(sam)) - {0} - set(rows["id"]))

    # Check 2: the truth under each SAM 3 outline equals what R16c step 5 saved for it
    under = pixels_under_outlines(sam, truth)[rows["id"].to_numpy()]
    differ = int(np.abs(under - rows[TRUTH_COLS].to_numpy()).sum())
    check(name, "2. truth under the outlines = R16c step 5's", f"{differ} of {int(under.sum()):,} px differ",
          differ <= LIMITS["truth_px_share"] * max(int(under.sum()), 1))

    DATA[name] = {"post": post, "pre": pre, "truth": truth, "sam": sam, "rows": rows, "pre_resized": pre_resized,
                  "ungraded": ungraded, "photos": "the DisasterM3 mirror" if ON_KAGGLE else
                  ("xBD's original files on this PC" if post is not None else "none on this PC")}
    print(f"  {name}: {shape[1]} x {shape[0]}, {len(rows)} graded outlines ({len(ungraded)} without a grade), "
          f"photos: {DATA[name]['photos']}{', before-photo resized' if pre_resized else ''}")

CHECK_DF = pd.DataFrame(CHECKS)
print(CHECK_DF.to_string(index=False))
if not CHECK_DF["pass"].all():
    raise RuntimeError("A check failed (table above), so nothing is written. Send Abrar this cell's output.")
print(f"✓ Checks 1 and 2 pass on {len(PAIRS)} pair(s)")

  hurricane_harvey_00000472_post_disaster.png: 1024 x 1024, 107 graded outlines (0 without a grade), photos: the DisasterM3 mirror
  santa_rosa_wildfire_00000058_post_disaster.png: 1024 x 1024, 290 graded outlines (0 without a grade), photos: the DisasterM3 mirror
  moore_tornado_00000120_post_disaster.png: 1024 x 1024, 23 graded outlines (0 without a grade), photos: the DisasterM3 mirror
  palu_tsunami_00000122_post_disaster.png: 1024 x 1024, 162 graded outlines (0 without a grade), photos: the DisasterM3 mirror
  midwest_flooding_00000287_post_disaster.png: 1024 x 1024, 9 graded outlines (0 without a grade), photos: the DisasterM3 mirror
  turkey_earthquake0_post_380.png: 1024 x 1024, 60 graded outlines (0 without a grade), photos: the DisasterM3 mirror
                                         image                                          check                                                              result  pass
   hurricane_harvey_00000472_post_disaster.png 1. truth = E1's sav

In [5]:
# ── Cell 5: The two maps per pair: R11 with TTA and find-then-grade; checks 3 and 4 ──
R11_COLS = ["r11_background_px", "r11_intact_px", "r11_damaged_px", "r11_destroyed_px"]

if ON_KAGGLE:
    import segmentation_models_pytorch as smp
    import torch
    import torch.nn.functional as F
    from huggingface_hub import HfApi, hf_hub_download

    def hf_token():
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            return os.environ.get("HF_TOKEN")

    TOKEN = hf_token()
    if not TOKEN:
        raise RuntimeError("Add the HF_TOKEN secret (AbrarAlam): R11's checkpoint is private.")
    REPO_ID = f"{HF_USER}/disasterm3-d4-{UNET}"
    SHA = HfApi().model_info(REPO_ID, token=TOKEN).sha
    UNET_CKPT = hf_hub_download(REPO_ID, "final_model.pth", revision=SHA, token=TOKEN)
    R11_CHECKPOINT = f"{REPO_ID}@{SHA[:8]}"
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # E7 step 2's code (Cell 6), unchanged: the D4 checkpoint, R11's inputs, the 8 views
    def load_unet(path):
        """D4 checkpoint: U-Net + ResNet-50, 6 input channels (after-image, then before-image), 4 classes."""
        state = torch.load(path, map_location=DEVICE, weights_only=False)
        if isinstance(state, dict) and "model_state_dict" in state:
            state = state["model_state_dict"]
        model = smp.Unet(encoder_name="resnet50", encoder_weights=None, in_channels=6, classes=NUM_CLASSES)
        model.load_state_dict(state)
        return model.to(DEVICE).eval()

    IMNET_MEAN = np.array([0.485, 0.456, 0.406], np.float32) * 255
    IMNET_STD = np.array([0.229, 0.224, 0.225], np.float32) * 255

    def unet_input(post, pre):
        """As R11's val_transform: ImageNet normalisation, channels 0-2 after-image, 3-5 before-image."""
        x = np.concatenate([(post.astype(np.float32) - IMNET_MEAN) / IMNET_STD,
                            (pre.astype(np.float32) - IMNET_MEAN) / IMNET_STD], 2)
        return torch.from_numpy(x.transpose(2, 0, 1).copy()).unsqueeze(0).to(DEVICE)

    def _forward_once(model, x):
        """Logits at x's own HxW: pad to a multiple of 32, exactly as in the D4 notebooks."""
        h, w = x.shape[-2:]
        ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
        if ph or pw:
            x = F.pad(x, (0, pw, 0, ph), mode="reflect")
        return model(x)[..., :h, :w].float()

    TTA_VIEWS = [(k, flip) for flip in (False, True) for k in range(4)]   # the D4 notebooks' 8 views

    def _tta_fwd(x, k, flip):
        x = torch.rot90(x, k, dims=(2, 3))
        return torch.flip(x, dims=(3,)) if flip else x

    def _tta_inv(y, k, flip):
        if flip:
            y = torch.flip(y, dims=(3,))
        return torch.rot90(y, -k, dims=(2, 3))

    @torch.no_grad()
    def predict_tta(model, x):
        """The mean of the 8 views' class probabilities, then the most likely class per pixel."""
        prob_sum = 0
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            for k, flip in TTA_VIEWS:
                logits = _forward_once(model, _tta_fwd(x, k, flip))
                prob_sum = prob_sum + torch.softmax(_tta_inv(logits, k, flip), dim=1)
        return (prob_sum / len(TTA_VIEWS)).argmax(1)[0].cpu().numpy().astype(np.uint8)

    unet = load_unet(UNET_CKPT)
    print(f"✓ {UNET} loaded: {R11_CHECKPOINT}, on {torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'}")
else:
    R11_CHECKPOINT = None
    print("✓ Dry run: E1's saved R9 maps stand in for R11 (no model on a PC)")


def same_iou(a, b, tol):
    return all((x is None) == (y is None) and (x is None or abs(x - y) <= tol) for x, y in zip(a, b))


MAPS, CHECKS = {}, []
for p in PAIRS:
    name, d = p["image"], DATA[p["image"]]
    if ON_KAGGLE:
        r11 = predict_tta(unet, unet_input(d["post"], d["pre"]))
    else:
        r11 = cv2.imread(str(R9_STAND_IN / name), cv2.IMREAD_GRAYSCALE)
        assert r11 is not None and r11.shape == d["truth"].shape, f"{name}: no stand-in map in {R9_STAND_IN}"
    ftg = find_then_grade_map(d["sam"], d["rows"], BIASES)
    r11_iou, ftg_iou = iou_per_class(r11, d["truth"]), iou_per_class(ftg, d["truth"])

    # Check 3: R11 reproduces its saved results (skipped in a dry run: there, R9 stands in for it)
    if ON_KAGGLE:
        check(name, "3a. R11's IoU per class = E5 step 2's (within 0.002)", f"{r11_iou} against {p['expect']['r11_iou']}",
              same_iou(r11_iou, p["expect"]["r11_iou"], LIMITS["r11_iou"]))
        under = pixels_under_outlines(d["sam"], r11)[d["rows"]["id"].to_numpy()]
        share = float(np.abs(under - d["rows"][R11_COLS].to_numpy()).sum()) / max(int(under.sum()), 1)
        check(name, "3b. R11's pixels under the outlines = R16c step 5's (within 2%)", f"{share:.4f} differ",
              share <= LIMITS["r11_px"])
    # Check 4: find-then-grade = the PC's computation from the same saved files
    check(name, "4. find-then-grade's IoU per class = the PC's", f"{ftg_iou} against {p['expect']['ftg_iou']}",
          same_iou(ftg_iou, p["expect"]["ftg_iou"], LIMITS["ftg_iou"]))
    MAPS[name] = {"before-after-unet": r11, "find-then-grade": ftg}

CHECK_DF = pd.concat([CHECK_DF, pd.DataFrame(CHECKS)], ignore_index=True)
print(CHECK_DF.to_string(index=False))
if not CHECK_DF["pass"].all():
    raise RuntimeError("A check failed (table above), so nothing is written. Send Abrar this cell's output.")
print(f"✓ Checks pass on {len(PAIRS)} pair(s){' (check 3 needs the GPU run)' if DRY_RUN else ''}")

final_model.pth: reconstructing file:   0%|          |  0.00B /  130MB            

final_model.pth: downloading bytes:           |  0.00B            

✓ R11 loaded: AbrarAlam/disasterm3-d4-R11@31974dac, on Tesla T4
                                         image                                                           check                                                                    result  pass
   hurricane_harvey_00000472_post_disaster.png                  1. truth = E1's saved truth (pixels per class)             [924432, 0, 109775, 14369] against [924432, 0, 109775, 14369]  True
   hurricane_harvey_00000472_post_disaster.png                     2. truth under the outlines = R16c step 5's                                                    0 of 103,939 px differ  True
santa_rosa_wildfire_00000058_post_disaster.png                  1. truth = E1's saved truth (pixels per class)             [744246, 233505, 0, 70825] against [744246, 233505, 0, 70825]  True
santa_rosa_wildfire_00000058_post_disaster.png                     2. truth under the outlines = R16c step 5's                                                    0 of 293,5

In [6]:
# ── Cell 6: The gallery files, in the contract's shape (T3_Website/DATA_CONTRACT.md) ──
SOURCES = {
    "before-after-unet": {"model": "A U-Net that reads the before and after images together", "tta": True,
                          "checkpoint": R11_CHECKPOINT},
    "find-then-grade": {"model": "SAM 3 outlines each building on the before image, and our grader (a Siamese ResNet-18 "
                                 "that compares the before and after crops) labels each outline",
                        "tta": False,
                        "checkpoint": f"grader_seed{TABLE_SEED}.pth, seed {TABLE_SEED}: its saved grades (Kaggle output "
                                      "r16c-step5-grader-on-sam3-outlines), on SAM 3's saved outlines (e3-sam3-building-finding)"},
}
if DRY_RUN:
    SOURCES["before-after-unet"] = {"model": "DRY RUN stand-in: the after-image-only U-Net's saved map (R9, with TTA), not R11",
                                    "tta": True, "checkpoint": None}


def contract_problems(pair_dir, method_dir, mask, rows, hs, truth):
    """The contract's rules this notebook can check; the website's validator checks everything again on the PC."""
    h, w = mask.shape
    out = []
    for b in rows:
        if b["area_px"] < MIN_AREA or b["bbox_x"] + b["bbox_w"] > w or b["bbox_y"] + b["bbox_h"] > h:
            out.append(f"building {b['building_id']}: area or bounding box")
        if not (b["bbox_x"] <= b["centroid_x"] <= b["bbox_x"] + b["bbox_w"] and b["bbox_y"] <= b["centroid_y"] <= b["bbox_y"] + b["bbox_h"]):
            out.append(f"building {b['building_id']}: centroid outside its box")
        if any(len(r) < 3 or any(not (0 <= x <= w and 0 <= y <= h) for x, y in r) for r in b["outline"]):
            out.append(f"building {b['building_id']}: outline")
    others = sum(b["class_id"] == 3 for b in rows)
    for b in rows:
        none_expected = others - (b["class_id"] == 3) == 0
        if (b["dist_to_nearest_destroyed_px"] is None) != none_expected:
            out.append(f"building {b['building_id']}: distance should be {'null' if none_expected else 'a number'}")
    for s in hs["hot_spots"]:
        if not (0 <= s["point_xy"][0] <= w and 0 <= s["point_xy"][1] <= h):
            out.append(f"hot spot {s['rank']} outside the image")
    for s in hs["safe_areas"]:
        if not (0 <= s["xy"][0] <= w and 0 <= s["xy"][1] <= h):
            out.append(f"safe area {s['rank']} outside the image")
    for kind in ("kde", "safe", "priority"):
        im = Image.open(method_dir / f"heat_{kind}.png")
        if im.mode != "L" or im.size[0] != im.size[1]:
            out.append(f"heat_{kind}.png isn't a square one-channel PNG")
    back = cv2.imread(str(method_dir / "mask.png"), cv2.IMREAD_UNCHANGED)
    if back is None or back.ndim != 2 or not np.array_equal(back, mask):
        out.append("mask.png doesn't read back as written")
    return out


SUMMARY, PROBLEMS = [], []
for p in PAIRS:
    name, d = p["image"], DATA[p["image"]]
    pair_id = re.sub(r"(_post_disaster)?\.png$", "", name)
    assert re.fullmatch(r"[a-z0-9_-]+", pair_id), pair_id
    pair_dir = OUT_DIR / pair_id
    pair_dir.mkdir(parents=True)
    event, source = event_and_source(name)
    h, w = d["truth"].shape
    pair = {"event": event, "source": source, "split": "validation, clean", "image": name, "width": w, "height": h}
    if p.get("hero"):
        pair["hero"] = True
    if DRY_RUN:
        pair["fixture"] = True
    write_json(pair_dir / "pair.json", pair)
    for photo, key in (("before", "pre"), ("after", "post")):
        if d[key] is not None:
            assert d[key].shape[:2] == (h, w), f"{name}: the {photo}-photo isn't {w} x {h}"
            Image.fromarray(d[key]).save(pair_dir / f"{photo}.webp", "WEBP", quality=WEBP_QUALITY)
    write_ids_png(pair_dir / "gt.png", d["truth"])

    for method, mask in MAPS[name].items():
        mdir = pair_dir / "methods" / method
        mdir.mkdir(parents=True)
        write_ids_png(mdir / "mask.png", mask)
        rows = buildings_from_mask(mask, name)
        write_json(mdir / "buildings.json", rows, compact=True)
        hs = heatmap_outputs(mask, d["truth"], mdir)          # also writes heat_kde.png, heat_safe.png, heat_priority.png
        write_json(mdir / "hotspots.json", hs)
        iou = iou_per_class(mask, d["truth"])
        write_json(mdir / "accuracy.json", {"iou": dict(zip(CLASS_KEYS, iou))})
        source_json = {**SOURCES[method], "notebook": NOTEBOOK, "date": TODAY}
        if DRY_RUN:
            source_json["fixture"] = True
        write_json(mdir / "source.json", source_json)
        PROBLEMS += [f"{pair_id}/{method}: {m}" for m in contract_problems(pair_dir, mdir, mask, rows, hs, d["truth"])]
        SUMMARY.append({"pair": pair_id, "method": method,
                        **{f"{c}_buildings": sum(b["class_id"] == i for b in rows) for i, c in CLASS_NAME.items()},
                        **{f"{c}_iou": v for c, v in zip(CLASS_KEYS, iou)},
                        "hot_spots": len(hs["hot_spots"]), "safe_areas": len(hs["safe_areas"]),
                        "buildings_json_kb": round((mdir / "buildings.json").stat().st_size / 1024, 1)})

SUMMARY_DF = pd.DataFrame(SUMMARY)
print(SUMMARY_DF.to_string(index=False))
if PROBLEMS:
    raise RuntimeError("Files break the contract:\n  " + "\n  ".join(PROBLEMS))
size_mb = sum(f.stat().st_size for f in OUT_DIR.rglob("*") if f.is_file()) / 1e6
print(f"✓ {len(PAIRS)} pair folder(s) written in {OUT_DIR} ({size_mb:.1f} MB); the contract's rules checked here hold")

                        pair            method  intact_buildings  damaged_buildings  destroyed_buildings  background_iou  intact_iou  damaged_iou  destroyed_iou  hot_spots  safe_areas  buildings_json_kb
   hurricane_harvey_00000472 before-after-unet                 0                 52                    0          0.9562         NaN       0.6615         0.0000          3           3               47.3
   hurricane_harvey_00000472   find-then-grade                 5                 73                    0          0.9546      0.0000       0.5871         0.0000          3           3               51.7
santa_rosa_wildfire_00000058 before-after-unet               100                  5                   39          0.9006      0.7950       0.0000         0.7517          3           3              109.7
santa_rosa_wildfire_00000058   find-then-grade               180                  2                   63          0.9234      0.8246       0.0000         0.7619          3           3     

In [7]:
# ── Cell 7: README, config and the zip ─────────────────────────────────────
import importlib.metadata as md


def version(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None


def fmt(v):
    return "–" if v is None or (isinstance(v, float) and math.isnan(v)) else f"{v:.4f}"


rows_md = []
for p in PAIRS:
    pair_id = re.sub(r"(_post_disaster)?\.png$", "", p["image"])
    s = SUMMARY_DF[SUMMARY_DF["pair"] == pair_id].set_index("method")
    rows_md.append(f"| {pair_id}{' (Home)' if p.get('hero') else ''} | {p['type']} | "
                   f"{fmt(s.loc['before-after-unet', 'destroyed_iou'])} | {fmt(s.loc['find-then-grade', 'destroyed_iou'])} | "
                   f"{fmt(s.loc['before-after-unet', 'background_iou'])} / {fmt(s.loc['before-after-unet', 'intact_iou'])} / "
                   f"{fmt(s.loc['before-after-unet', 'damaged_iou'])} | "
                   f"{int(s.loc['before-after-unet', 'intact_buildings'] + s.loc['before-after-unet', 'damaged_buildings'] + s.loc['before-after-unet', 'destroyed_buildings'])} / "
                   f"{int(s.loc['find-then-grade', 'intact_buildings'] + s.loc['find-then-grade', 'damaged_buildings'] + s.loc['find-then-grade', 'destroyed_buildings'])} |")
check_md = [f"| {r.image} | {r.check} | {r.result} | {'pass' if r['pass'] else 'FAIL'} |" for _, r in CHECK_DF.iterrows()]

readme = f"""# T3 gallery export{' (DRY RUN on a PC: not for the website)' if DRY_RUN else ''}

Run on {TODAY}{' with SMOKE_TEST' if SMOKE_TEST else ''}. Notebook: `{NOTEBOOK}`. The rule that chose the pairs, and
why, is in the notebook's first cell.

## The pairs

| Pair | Type | Destroyed IoU, before-after U-Net | Destroyed IoU, find-then-grade | U-Net background / intact / damaged IoU | Buildings (U-Net / find-then-grade) |
|---|---|---|---|---|---|
{chr(10).join(rows_md)}

IoU: the overlap between a map's pixels of a class and the true pixels of it, 0 to 1; "–" where the class is in
neither. Buildings: the table's rows, 8-connected regions of one class of at least {MIN_AREA} px.

## Checks

| Pair | Check | Result | |
|---|---|---|---|
{chr(10).join(check_md)}

## Files

- `gallery/<pair>/`: one folder per pair, as in `Essential_task_delegation/T3_Website/DATA_CONTRACT.md`.
- `t3_gallery.zip`: the same folders, to unzip into `T3_Website/public/data/`.
- `README.md` (this file) and `config.json`: the settings, the pairs, the checks and the versions.

## Using it

1. Unzip `t3_gallery.zip` into `Essential_task_delegation/T3_Website/public/data/`, so that
   `public/data/santa_rosa_wildfire_00000058/pair.json` exists.
2. In `T3_Website/`: `npm run validate` (the website's full check), then `npm run build`.
"""
(OUT_ROOT / "README.md").write_text(readme, encoding="utf-8")

config = {
    "date": TODAY, "dry_run": DRY_RUN, "smoke_test": SMOKE_TEST, "notebook": NOTEBOOK,
    "rule": "605 clean validation pairs with at least 5,000 true destroyed px; one event per disaster type (most candidates); "
            "per event the median of R11's destroyed IoU (E5 step 2, TTA); Home: the largest destroyed area outside Turkey",
    "pairs": PAIRS, "methods": SOURCES, "table_seed": TABLE_SEED, "biases": dict(zip(LABELS, BIASES)),
    "settings": {"min_area_px": MIN_AREA, "outline_tolerance_px": OUTLINE_TOLERANCE_PX, "webp_quality": WEBP_QUALITY,
                 "heatmap": {"sigma_px": SIGMA_PX, "damaged_weight": DAMAGED_WEIGHT, "safe_scale_px": SAFE_SCALE_PX,
                             "hot_threshold": HOT_THRESHOLD, "min_spot_px": MIN_SPOT_PX,
                             "point_edge_margin_px": POINT_EDGE_MARGIN_PX, "safe_min_gap_px": SAFE_MIN_GAP_PX,
                             "safe_edge_margin_px": SAFE_EDGE_MARGIN_PX, "point_radius_px": POINT_RADIUS_PX}},
    "limits": LIMITS, "inputs": {"sam3_maps": str(E3_DIR), "grades": str(INST_PATH), "biases": str(CALIB_PATH)},
    "checks": CHECK_DF.to_dict("records"), "summary": SUMMARY_DF.to_dict("records"),
    "photos": {p["image"]: DATA[p["image"]]["photos"] for p in PAIRS},
    "before_photo_resized": [p["image"] for p in PAIRS if DATA[p["image"]]["pre_resized"]],
    "versions": {k: version(k) for k in ("numpy", "opencv-python", "opencv-python-headless", "scipy", "pandas", "pillow",
                                         "torch", "segmentation-models-pytorch", "huggingface-hub")},
}
write_json(OUT_ROOT / "config.json", json.loads(json.dumps(config, default=str)))

zip_path = OUT_ROOT / "t3_gallery.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT_DIR.rglob("*")):
        if f.is_file():
            z.write(f, f.relative_to(OUT_DIR).as_posix())
print(readme)
print(f"✓ Wrote {OUT_ROOT / 'README.md'}, {OUT_ROOT / 'config.json'} and {zip_path} "
      f"({zip_path.stat().st_size / 1e6:.1f} MB){'  [DRY RUN: not for the website]' if DRY_RUN else ''}")

# T3 gallery export

Run on 2026-11-17. Notebook: `Essential_task_delegation/Champion_Task/t3-gallery-export.ipynb`. The rule that chose the pairs, and
why, is in the notebook's first cell.

## The pairs

| Pair | Type | Destroyed IoU, before-after U-Net | Destroyed IoU, find-then-grade | U-Net background / intact / damaged IoU | Buildings (U-Net / find-then-grade) |
|---|---|---|---|---|---|
| hurricane_harvey_00000472 | hurricane | 0.0000 | 0.0000 | 0.9562 / – / 0.6615 | 52 / 78 |
| santa_rosa_wildfire_00000058 (Home) | wildfire | 0.7517 | 0.7619 | 0.9006 / 0.7950 / 0.0000 | 144 / 245 |
| moore_tornado_00000120 | tornado | 0.6592 | 0.5311 | 0.9928 / 0.0000 / 0.4001 | 21 / 20 |
| palu_tsunami_00000122 | tsunami | 0.2769 | 0.3142 | 0.8927 / 0.6914 / 0.0000 | 107 / 106 |
| midwest_flooding_00000287 | flood | 0.0000 | 0.0000 | 0.9716 / – / 0.0000 | 8 / 2 |
| turkey_earthquake0_post_380 | earthquake | 0.0000 | 0.4596 | 0.8511 / 0.6495 / – | 42 / 34 |

IoU: the overlap between a map's pixe